# Prueba de la API de NYC 311
### Descarga de enero 2025 y primera mirada al tiempo de cierre de los casos

## 1. Imports y configuración inicial

In [21]:
import os

import pandas as pd
import requests
from dotenv import load_dotenv

In [22]:
load_dotenv()
token = os.getenv("NYC_APP_TOKEN")
len(token)

25

In [23]:
url = "https://data.cityofnewyork.us/resource/erm2-nwe9.json"

## 2. Primera llamada a la API

In [24]:
params = {"$limit": 5}
headers = {"X-App-Token": token}

In [25]:
r = requests.get(url=url, params=params, headers=headers, timeout=30)
print(r.status_code)

200


In [26]:
df = pd.DataFrame(r.json())
df.shape

(5, 36)

In [27]:
df.head()

,unique_key,created_date,agency,agency_name,complaint_type,descriptor,location_type,incident_zip,incident_address,street_name,...,park_facility_name,park_borough,vehicle_type,latitude,longitude,location,:@computed_region_f5dn_yrer,:@computed_region_yeji_bk3q,:@computed_region_sbqj_enih,:@computed_region_92fq_4b7q
0,70956561,2026-11-01T02:06:29.000,NYPD,New York City Police Department,Noise - Vehicle,Car/Truck Music,Street/Sidewalk,11368,127-11 WILLETS POINT BOULEVARD,WILLETS POINT BOULEVARD,...,Unspecified,QUEENS,Car,40.758650131211176,-73.84174301320034,"{'type': 'Point', 'coordinates': [-73.84174301...",22,3,68,21
1,70966697,2026-11-01T02:06:11.000,NYPD,New York City Police Department,Noise - Street/Sidewalk,Loud Music/Party,Street/Sidewalk,11368,127-11 WILLETS POINT BOULEVARD,WILLETS POINT BOULEVARD,...,Unspecified,QUEENS,NaN,40.758650131211176,-73.84174301320034,"{'type': 'Point', 'coordinates': [-73.84174301...",22,3,68,21
2,70968101,2026-11-01T02:06:09.000,NYPD,New York City Police Department,Noise - Street/Sidewalk,Loud Talking,Street/Sidewalk,11433,106-60 GUY R BREWER BOULEVARD,GUY R BREWER BOULEVARD,...,Unspecified,QUEENS,NaN,40.69882468593988,-73.79291016411402,"{'type': 'Point', 'coordinates': [-73.79291016...",41,3,61,46
3,70958356,2026-11-01T02:05:30.000,NYPD,New York City Police Department,Noise - Street/Sidewalk,Loud Music/Party,Street/Sidewalk,10002,188 ORCHARD STREET,ORCHARD STREET,...,Unspecified,MANHATTAN,NaN,40.722069137616074,-73.98818856611241,"{'type': 'Point', 'coordinates': [-73.98818856...",70,4,4,32
4,70963675,2026-11-01T02:05:30.000,NYPD,New York City Police Department,Noise - Street/Sidewalk,Loud Music/Party,Street/Sidewalk,11368,127-11 WILLETS POINT BOULEVARD,WILLETS POINT BOULEVARD,...,Unspecified,QUEENS,NaN,40.758650131211176,-73.84174301320034,"{'type': 'Point', 'coordinates': [-73.84174301...",22,3,68,21


In [28]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 36 columns):
 #   Column                       Non-Null Count  Dtype 
---  ------                       --------------  ----- 
 0   unique_key                   5 non-null      object
 1   created_date                 5 non-null      object
 2   agency                       5 non-null      object
 3   agency_name                  5 non-null      object
 4   complaint_type               5 non-null      object
 5   descriptor                   5 non-null      object
 6   location_type                5 non-null      object
 7   incident_zip                 5 non-null      object
 8   incident_address             5 non-null      object
 9   street_name                  5 non-null      object
 10  cross_street_1               5 non-null      object
 11  cross_street_2               5 non-null      object
 12  intersection_street_1        5 non-null      object
 13  intersection_street_2        5 non-null

## 3. Consulta de un mes 

In [29]:
where = (
    "complaint_type = 'Noise - Residential' "
    "AND created_date >= '2025-01-01T00:00:00' "
    "AND created_date < '2025-02-01T00:00:00'"
)

columnas = [
    "unique_key",
    "created_date",
    "closed_date",
    "descriptor",
    "incident_zip",
    "borough",
    "police_precinct",
    "open_data_channel_type",
    "status",
    "latitude",
    "longitude",
]

params = {
    "$where": where,
    "$select": ",".join(columnas),
    "$order": "unique_key",
    "$limit": 50_000,
    "$offset": 0,
}


r = requests.get(url=url, params=params, headers=headers, timeout=30)
print(r.status_code)

df = pd.DataFrame(r.json())
df.shape

200


(50000, 11)

### Conteo de casos reales

In [30]:
params_conteo = {"$select": "count(*)", "$where": where}

r_conteo = requests.get(url, params=params_conteo, headers=headers, timeout=30)
r_conteo.json()

[{'count': '70837'}]

### Paginación 

In [31]:
limit = 50_000
offset = 0
paginas = []

while True:
    params = {
        "$where": where,
        "$select": ",".join(columnas),
        "$order": "unique_key",
        "$limit": limit,
        "$offset": offset,
    }
    r = requests.get(url, params=params, headers=headers, timeout=60)
    r.raise_for_status()

    pagina = pd.DataFrame(r.json())
    paginas.append(pagina)
    print(f"offset={offset}, recibi {len(pagina)} filas")

    if len(pagina) < limit:
        break

    offset += limit

df = pd.concat(paginas, ignore_index=True)
df.shape

offset=0, recibi 50000 filas
offset=50000, recibi 20837 filas


(70837, 11)

In [32]:
print(df["unique_key"].is_unique)
print(df["created_date"].min())
print(df["created_date"].max())


True
2025-01-01T00:00:12.000
2025-01-31T23:58:47.000


In [33]:
df.dtypes

unique_key                object
created_date              object
closed_date               object
descriptor                object
incident_zip              object
borough                   object
police_precinct           object
open_data_channel_type    object
status                    object
latitude                  object
longitude                 object
dtype: object

### Conversión de datos

In [34]:
df["created_date"] = pd.to_datetime(df["created_date"])
df["closed_date"] = pd.to_datetime(df["closed_date"])
df["latitude"] = pd.to_numeric(df["latitude"], errors="coerce")
df["longitude"] = pd.to_numeric(df["longitude"], errors="coerce")

In [35]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 70837 entries, 0 to 70836
Data columns (total 11 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   unique_key              70837 non-null  object        
 1   created_date            70837 non-null  datetime64[ns]
 2   closed_date             70837 non-null  datetime64[ns]
 3   descriptor              70837 non-null  object        
 4   incident_zip            70836 non-null  object        
 5   borough                 70837 non-null  object        
 6   police_precinct         70837 non-null  object        
 7   open_data_channel_type  70837 non-null  object        
 8   status                  70837 non-null  object        
 9   latitude                70835 non-null  float64       
 10  longitude               70835 non-null  float64       
dtypes: datetime64[ns](2), float64(2), object(7)
memory usage: 5.9+ MB


### Tiempos de cierre

In [36]:
df["horas_cierre"] = (df["closed_date"] - df["created_date"]).dt.total_seconds() / 3600

In [37]:
df["horas_cierre"].describe().round(2)

count    70837.00
mean         6.84
std          8.24
min          0.00
25%          0.96
50%          3.50
75%         10.36
max        119.04
Name: horas_cierre, dtype: float64

In [38]:
(df["horas_cierre"] < 1 / 60).sum()

np.int64(839)

## 4. Descarga con la función de `dataset.py`

La lógica de las secciones 2 y 3 quedó en `descargar_mes`. Aquí se comprueba que da el mismo resultado y que el Parquet se guarda bien.

In [ ]:
%load_ext autoreload
%autoreload 2

from nyc311_operational_analytics.config import RAW_DIR
from nyc311_operational_analytics.dataset import descargar_mes

In [40]:
df_mes = descargar_mes(2025, 1)
df_mes.shape

offset=0, 50000 filas recibidas
offset=50000, 20837 filas recibidas
Guardado 2025-01.parquet: 70837 filas


(70837, 11)

In [41]:
df_mes.dtypes

unique_key                        object
created_date              datetime64[ns]
closed_date               datetime64[ns]
descriptor                        object
incident_zip                      object
borough                           object
police_precinct                   object
latitude                         float64
longitude                        float64
open_data_channel_type            object
status                            object
dtype: object

In [42]:
guardado = pd.read_parquet(RAW_DIR / "2025-01.parquet")

print(guardado.shape)
print(guardado.dtypes.equals(df_mes.dtypes))

(70837, 11)
True


## 5. Hallazgos

**Sobre la API**
- Enero de 2025 tiene 70.837 solicitudes de *Noise - Residential*, no unas 38.000 como estimaba el ROADMAP. No cabe en una página de 50.000 filas: la paginación es obligatoria.
- El JSON omite los campos vacíos: sin `$select`, la cantidad de columnas depende de las filas (36 en la prueba de 5). Con `$select` siempre llegan las 11.
- Todo llega como texto. Las fechas son *floating timestamps*, sin zona horaria (hora local de Nueva York).

**Sobre los datos de enero**
- Nulos: 0 en `closed_date`, 1 en `incident_zip` y 2 en `latitude` y `longitude`.
- El tiempo de cierre está sesgado a la derecha: la media (6.84 h) casi duplica a la mediana (3.50 h).
- La mitad central de los casos cierra entre 0.96 y 10.4 h. El máximo es de 119 h (unos 5 días); ninguno pasa de 7 días.
- No hay cierres en 0 h ni negativos, pero el mínimo es de 4 segundos y 839 casos (1.2 %) cierran en menos de 1 minuto. Hay que revisarlos en el EDA: pueden ser cierres automáticos.
- El P75 de enero es 10.4 h, pero no es el umbral de cierre tardío: ese se calcula con enero–agosto.